In [34]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, roc_auc_score)

In [35]:
df = pd.read_csv('/kaggle/input/datasets/ozfmomani/hotel-booking/hotel-booking.csv')

df.columns = df.columns.str.strip()

print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nFirst 5 rows:")
print(df.head())
print("\nNumerical stats:")
print(df.describe().T)
print("\nCategorical stats:")
print(df.describe(include='object').T)
print("\nTarget distribution:")
print(df['booking status'].value_counts())
print("\nCancellation rate:", round(df['booking status'].eq('Canceled').mean() * 100, 2), "%")

Shape: (36285, 17)

Dtypes:
 Booking_ID                   object
number of adults              int64
number of children            int64
number of weekend nights      int64
number of week nights         int64
type of meal                 object
car parking space             int64
room type                    object
lead time                     int64
market segment type          object
repeated                      int64
P-C                           int64
P-not-C                       int64
average price               float64
special requests              int64
date of reservation          object
booking status               object
dtype: object

First 5 rows:
  Booking_ID  number of adults  number of children  number of weekend nights  \
0   INN00001                 1                   1                         2   
1   INN00002                 1                   0                         1   
2   INN00003                 2                   1                         1   
3   INN000

In [36]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicated rows:", df.duplicated().sum())

print("\nSuspicious values:")
print("average price = 0:", (df['average price'] == 0).sum())
print("number of adults = 0:", (df['number of adults'] == 0).sum())
print("number of adults = 0 AND number of children = 0:",
      ((df['number of adults'] == 0) & (df['number of children'] == 0)).sum())
print("number of children > 3:", (df['number of children'] > 3).sum())

print("\nOutlier check (IQR method):")
num_cols = ['lead time', 'average price', 'number of adults', 'number of children',
            'number of weekend nights', 'number of week nights', 'P-C', 'P-not-C']
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    print(f"  {col}: {outliers} outliers (lower: {lower:.2f}, upper: {upper:.2f})")

Missing values:
Booking_ID                  0
number of adults            0
number of children          0
number of weekend nights    0
number of week nights       0
type of meal                0
car parking space           0
room type                   0
lead time                   0
market segment type         0
repeated                    0
P-C                         0
P-not-C                     0
average price               0
special requests            0
date of reservation         0
booking status              0
dtype: int64

Duplicated rows: 0

Suspicious values:
average price = 0: 545
number of adults = 0: 139
number of adults = 0 AND number of children = 0: 0
number of children > 3: 3

Outlier check (IQR method):
  lead time: 1332 outliers (lower: -146.50, upper: 289.50)
  average price: 1696 outliers (lower: 20.75, upper: 179.55)
  number of adults: 10175 outliers (lower: 2.00, upper: 2.00)
  number of children: 2702 outliers (lower: 0.00, upper: 0.00)
  number of weekend n

In [37]:
df_clean = df.copy()


df_clean['is_zero_price'] = (df_clean['average price'] == 0).astype(int)
print("Zero price flagged:", df_clean['is_zero_price'].sum())


cols_to_cap = ['lead time', 'average price', 'number of week nights']
for col in cols_to_cap:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df_clean[col] = df_clean[col].clip(lower=lower, upper=upper)
    print(f"{col} capped at [{lower:.2f}, {upper:.2f}]")

Zero price flagged: 545
lead time capped at [-146.50, 289.50]
average price capped at [20.75, 179.55]
number of week nights capped at [-2.00, 6.00]


In [38]:
df_corr = df_clean.copy()
df_corr['canceled'] = (df_corr['booking status'] == 'Canceled').astype(int)

num_cols = ['number of adults', 'number of children', 'number of weekend nights',
            'number of week nights', 'car parking space', 'lead time', 'repeated',
            'P-C', 'P-not-C', 'average price', 'special requests', 'is_zero_price']

print("Correlation with cancellation (sorted):")
print(df_corr[num_cols + ['canceled']].corr()['canceled'].drop('canceled').sort_values(ascending=False).round(3))

Correlation with cancellation (sorted):
lead time                   0.442
average price               0.143
number of adults            0.087
number of week nights       0.085
number of weekend nights    0.061
number of children          0.033
P-C                        -0.034
P-not-C                    -0.060
is_zero_price              -0.083
car parking space          -0.086
repeated                   -0.107
special requests           -0.253
Name: canceled, dtype: float64


In [39]:
df_feat = df_clean.copy()


df_feat['total_nights'] = df_feat['number of weekend nights'] + df_feat['number of week nights']

df_feat['total_guests'] = df_feat['number of adults'] + df_feat['number of children']

df_feat['is_long_lead'] = (df_feat['lead time'] > 90).astype(int)


df_feat['canceled'] = (df_feat['booking status'] == 'Canceled').astype(int)
new_feats = ['total_nights', 'total_guests', 'is_long_lead']
print("New features correlation with cancellation:")
print(df_feat[new_feats + ['canceled']].corr()['canceled'].drop('canceled').sort_values(ascending=False).round(3))

New features correlation with cancellation:
is_long_lead    0.381
total_nights    0.099
total_guests    0.090
Name: canceled, dtype: float64


In [40]:

drop_cols = ['Booking_ID', 'date of reservation', 'number of adults', 'number of children',
             'number of weekend nights', 'number of week nights', 'P-C', 'P-not-C', 'canceled']
df_model = df_feat.drop(columns=drop_cols)

y = (df_feat['booking status'] == 'Canceled').astype(int)
df_model = df_model.drop(columns=['booking status'])


cat_cols = ['type of meal', 'room type', 'market segment type']
df_model = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

print("Final shape:", df_model.shape)
print("\nColumns:")
print(df_model.columns.tolist())

Final shape: (36285, 22)

Columns:
['car parking space', 'lead time', 'repeated', 'average price', 'special requests', 'is_zero_price', 'total_nights', 'total_guests', 'is_long_lead', 'type of meal_Meal Plan 2', 'type of meal_Meal Plan 3', 'type of meal_Not Selected', 'room type_Room_Type 2', 'room type_Room_Type 3', 'room type_Room_Type 4', 'room type_Room_Type 5', 'room type_Room_Type 6', 'room type_Room_Type 7', 'market segment type_Complementary', 'market segment type_Corporate', 'market segment type_Offline', 'market segment type_Online']


In [41]:
X = df_model
y = (df_feat['booking status'] == 'Canceled').astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = ['lead time', 'average price', 'special requests',
            'total_nights', 'total_guests']

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("\nTarget distribution in train:")
print(y_train.value_counts(normalize=True).round(3))
print("\nTarget distribution in test:")
print(y_test.value_counts(normalize=True).round(3))

X_train shape: (29028, 22)
X_test shape: (7257, 22)

Target distribution in train:
booking status
0    0.672
1    0.328
Name: proportion, dtype: float64

Target distribution in test:
booking status
0    0.672
1    0.328
Name: proportion, dtype: float64


In [42]:
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    'XGBoost': XGBClassifier(scale_pos_weight=2, random_state=42, eval_metric='logloss')
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = accuracy_score(y_test, y_pred)
    print(f"\n--- {name} ---")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(classification_report(y_test, y_pred, target_names=['Not Canceled', 'Canceled']))

print("\nModel Accuracy Summary:")
for name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name}: {acc:.4f}")


--- Logistic Regression ---
Accuracy: 0.7842
              precision    recall  f1-score   support

Not Canceled       0.88      0.79      0.83      4879
    Canceled       0.64      0.77      0.70      2378

    accuracy                           0.78      7257
   macro avg       0.76      0.78      0.77      7257
weighted avg       0.80      0.78      0.79      7257


--- KNN ---
Accuracy: 0.8554
              precision    recall  f1-score   support

Not Canceled       0.88      0.91      0.89      4879
    Canceled       0.80      0.75      0.77      2378

    accuracy                           0.86      7257
   macro avg       0.84      0.83      0.83      7257
weighted avg       0.85      0.86      0.85      7257


--- Random Forest ---
Accuracy: 0.8877
              precision    recall  f1-score   support

Not Canceled       0.90      0.93      0.92      4879
    Canceled       0.85      0.79      0.82      2378

    accuracy                           0.89      7257
   macro avg